In [0]:
from pyspark.sql.functions import col, count, round, sum

orders = spark.table("silver.olist.orders")
payments = spark.table("silver.olist.order_payments")
customers = spark.table("silver.olist.customers")

# Join orders + payments + customers
df = payments \
    .join(orders.select("order_id", "customer_id"), on="order_id", how="inner") \
    .join(customers.select("customer_id", "customer_state", "customer_city"), on="customer_id", how="left")

# Aggregate payment preferences by region
gold_payments = df.groupBy("customer_state", "payment_type") \
    .agg(
        count("order_id").alias("total_transactions"),
        round(sum("payment_value"), 2).alias("total_payment_value")
    ) \
    .orderBy(col("total_payment_value").desc())

# Save to Gold Layer
gold_payments.write.format("delta").mode("overwrite") \
    .option("path", "abfss://gold@project1storagek.dfs.core.windows.net/payment_methods_by_state") \
    .saveAsTable("gold.olist.payment_methods_by_state")

print("payment_methods_by_state done:", gold_payments.count())
gold_payments.show(10)

payment_methods_by_state done: 106
+--------------+------------+------------------+-------------------+
|customer_state|payment_type|total_transactions|total_payment_value|
+--------------+------------+------------------+-------------------+
|            SP| credit_card|             32168|         4676662.34|
|            RJ| credit_card|             10288|         1730340.68|
|            MG| credit_card|              9070|         1471975.31|
|            SP|      boleto|              8205|         1084604.44|
|            RS| credit_card|              3985|          662551.39|
|            PR| credit_card|              3786|          626606.74|
|            BA| credit_card|              2662|          498907.13|
|            SC| credit_card|              2713|          481784.37|
|            MG|      boleto|              2304|           340518.7|
|            RJ|      boleto|              2163|          328621.78|
+--------------+------------+------------------+-------------------+

In [0]:
spark.sql("DROP TABLE IF EXISTS gold.olist.payment_methods_by_state")

DataFrame[]